# 02b — Khôi phục các bài Lao Động lưu nhầm trang cookie
Chạy trên **CPU** sau notebook 01 và 02 của Stage A. Notebook đọc raw `stage-a-v2`, chỉ lấy lại những bản ghi Lao Động có trang cookie D1N, giữ nguyên các ID còn lại, rồi xây một raw mới giữ đủ 1.000 outcome; số documents phụ thuộc parse. Raw và build cũ không bị sửa.

Chỉ kết quả được adapter xác nhận có đúng article ID, tiêu đề và thân bài mới thay thế raw cũ. Nếu trang tiếp tục lỗi, bản ghi cũ được giữ để báo cáo parse failure.


Notebook tùy chọn, không thuộc luồng chính 00–04. Bootstrap dùng optional_data_code_lock.json riêng. Runtime mới chạy lại từ đầu để resume. Khi chạy team, cả ba ghim cùng full SHA vào CODE_REVISION trước khi crawl. Build bằng code đã merge dùng tên run mới; notebook 03 phải dùng cùng full CODE_COMMIT khi freeze.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "optional_data_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "optional_data_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


In [ ]:
from vietmedbridge.artifacts import read_json
from vietmedbridge.crawl import CrawlConfig, read_completed_crawl
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.parse_recovery import recover_laodong_challenges
SNAPSHOT = load_snapshot(DATA_ROOT)
PILOT = read_json(DATA_ROOT / 'reports/inventory/stage_a.json')
SOURCE_CRAWL = 'stage-a-v2'
RECOVERED_CRAWL = 'stage-a-laodong-recovered-reviewed-v1'
NEW_BUILD = 'stage-a-data-v3-laodong-reviewed'
BASELINE = read_completed_crawl(
    DATA_ROOT / 'crawl' / SOURCE_CRAWL,
    links_path=DATA_ROOT / PILOT['files']['stage_a_links']['path'],
    origin_corpus_sha256=SNAPSHOT['files']['links_corpus.parquet']['sha256'],
)
print('Bản ghi gốc:', BASELINE['recorded_documents'], '| HTTP ok:', BASELINE['statuses'].get('ok', 0))
RECOVERY = await recover_laodong_challenges(
    DATA_ROOT / 'crawl' / SOURCE_CRAWL, DATA_ROOT / 'crawl',
    run_name=RECOVERED_CRAWL, work_dir=WORK_DIR,
    config=CrawlConfig(concurrency=2, per_host_delay=2.0, attempts=2),
)
print(json.dumps(RECOVERY, ensure_ascii=False, indent=2))


In [ ]:
from vietmedbridge.build import build_corpus
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.health import health_report
TOKENIZER_LOCK = read_json(DATA_ROOT / 'tokenizer_lock.json')
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(TOKENIZER_LOCK['revision'])
PIPELINE_CONFIG = json.loads((CHECKOUT / 'configs/data_pipeline.json').read_text())
BUILD = build_corpus(
    DATA_ROOT / 'crawl' / RECOVERED_CRAWL, DATA_ROOT / 'processed',
    TOKENIZER, TOKENIZER_SPEC, run_name=NEW_BUILD,
    config=ChunkConfig(**PIPELINE_CONFIG['chunking']), work_dir=WORK_DIR,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
)
print('Build counts:', json.dumps(BUILD['counts'], ensure_ascii=False, indent=2))
HEALTH = health_report(
    DATA_ROOT / 'processed' / NEW_BUILD,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
    crawl_dir=DATA_ROOT / 'crawl' / RECOVERED_CRAWL,
    work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG['audit_size'],
)
print('Coverage:', json.dumps(HEALTH['coverage'], ensure_ascii=False, indent=2))
print('Lao Động:', [row for row in HEALTH['by_domain'] if 'laodong.vn' in row.get('domain', '')])


So sánh recovered_articles/documents/parse_failed với baseline đã lưu. Để freeze, mở notebook 03 trong runtime mới, đặt CODE_REVISION thành full CODE_COMMIT in ở Bootstrap, CRAWL_RUN=RECOVERED_CRAWL và BUILD_RUN=NEW_BUILD theo các tên đã chọn ở notebook này.